# NUS DSS5102 — Advanced Regression and Time Series Analysis
## A methodical, student-oriented sustainability modelling notebook

This notebook is a **teaching companion** for the major ideas associated with NUS DSS5102. It deliberately connects the topics rather than presenting them as isolated algorithms.

### Big picture

We begin with a familiar regression question:

> **How can we explain and predict campus electricity consumption?**

Then we progressively discover why ordinary linear regression is insufficient:

$$
\text{Linear regression}
\rightarrow
\text{GLMs}
\rightarrow
\text{regularisation + resampling}
\rightarrow
\text{splines / local regression / GAM-style models}
\rightarrow
\text{mixed effects}
\rightarrow
\text{ARIMA / VAR / state-space models}.
$$

### Core learning objectives

By the end, you should be able to:

1. distinguish **conditional mean modelling** from **distributional modelling**;
2. use logistic and Poisson **generalized linear models**;
3. explain the **bias–variance trade-off** and use cross-validation;
4. understand Ridge/Lasso shrinkage;
5. use the **bootstrap** to quantify uncertainty computationally;
6. model nonlinear effects with splines, LOWESS and additive smooths;
7. recognize grouped dependence and fit a **mixed-effects model**;
8. diagnose time-series dependence using ACF/PACF and stationarity tests;
9. fit and evaluate ARIMA-type forecasts;
10. understand multivariate dynamics using VAR;
11. understand latent-state modelling and the Kalman-filter viewpoint;
12. compare models and construct a simple **model average**.

> **Pedagogical note.** We use a reproducible synthetic sustainability case study. Because we know how the data were generated, we can distinguish genuine structure from modelling artefacts. The workflow can later be transferred to real energy, climate, emissions or demand data.

# 0. Setup and reproducibility

The notebook uses:

- `numpy` / `pandas` for numerical and tabular work;
- `statsmodels` for statistical inference, GLMs, mixed models and time series;
- `scikit-learn` for preprocessing, cross-validation and regularisation;
- `Bokeh` for **all visualisations**.

The `Config` object keeps experimental settings in one place. This is useful in real projects because changing a random seed, forecast horizon or bootstrap count should not require editing many cells.

In [1]:
from __future__ import annotations

from dataclasses import dataclass
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

from scipy.special import expit
from scipy.stats import norm

import statsmodels.api as sm
import statsmodels.formula.api as smf
from statsmodels.nonparametric.smoothers_lowess import lowess
from statsmodels.tsa.stattools import acf, pacf, adfuller
from statsmodels.tsa.arima.model import ARIMA
from statsmodels.tsa.api import VAR
from statsmodels.tsa.statespace.structural import UnobservedComponents

from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import KFold, cross_validate
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, PolynomialFeatures, StandardScaler, SplineTransformer

from bokeh.io import output_notebook, show
from bokeh.layouts import column, gridplot
from bokeh.models import Band, ColumnDataSource, HoverTool, Span
from bokeh.plotting import figure

output_notebook()

@dataclass(frozen=True)
class Config:
    seed: int = 5102
    n_days: int = 730
    n_buildings: int = 8
    test_days: int = 90
    bootstrap_reps: int = 250
    cv_splits: int = 5

CFG = Config()
RNG = np.random.default_rng(CFG.seed)

print(CFG)

Loading BokehJS ...

Config(seed=5102, n_days=730, n_buildings=8, test_days=90, bootstrap_reps=250, cv_splits=5)


## 0.1 Reusable helpers

A recurring error in analytical notebooks is copying plotting and evaluation code everywhere. That makes later fixes painful and introduces inconsistencies.

We therefore centralise common operations:

- `regression_metrics()` computes comparable predictive metrics;
- `make_time_plot()` and `acf_plot()` standardise Bokeh output;
- `safe_mape()` avoids division-by-zero problems.

This is not merely software engineering polish: **consistent evaluation is part of valid statistical comparison**.

In [2]:
def rmse(y_true, y_pred) -> float:
    return float(np.sqrt(mean_squared_error(y_true, y_pred)))


def safe_mape(y_true, y_pred, eps: float = 1e-8) -> float:
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    denom = np.maximum(np.abs(y_true), eps)
    return float(np.mean(np.abs((y_true - y_pred) / denom)) * 100)


def regression_metrics(y_true, y_pred, model_name: str) -> dict:
    return {
        "model": model_name,
        "RMSE": rmse(y_true, y_pred),
        "MAE": float(mean_absolute_error(y_true, y_pred)),
        "R2": float(r2_score(y_true, y_pred)),
        "MAPE_%": safe_mape(y_true, y_pred),
    }


def make_time_plot(df, x, ys, title, y_label, width=900, height=340):
    p = figure(
        x_axis_type="datetime",
        width=width,
        height=height,
        title=title,
        tools="pan,wheel_zoom,box_zoom,reset,save",
    )
    for col_name, legend_name in ys.items():
        p.line(df[x], df[col_name], line_width=2, legend_label=legend_name)
    p.xaxis.axis_label = "Date"
    p.yaxis.axis_label = y_label
    p.legend.location = "top_left"
    p.legend.click_policy = "hide"
    return p


def acf_plot(values, nlags=30, partial=False, title=None):
    values = pd.Series(values).dropna().astype(float)
    corr = pacf(values, nlags=nlags, method="ywm") if partial else acf(values, nlags=nlags, fft=True)
    lags = np.arange(len(corr))
    bound = 1.96 / np.sqrt(len(values))

    p = figure(width=850, height=300, title=title or ("PACF" if partial else "ACF"))
    p.segment(lags, 0, lags, corr, line_width=2)
    p.scatter(lags, corr, size=6)
    p.add_layout(Span(location=bound, dimension="width", line_dash="dashed"))
    p.add_layout(Span(location=-bound, dimension="width", line_dash="dashed"))
    p.xaxis.axis_label = "Lag"
    p.yaxis.axis_label = "Correlation"
    return p

# 1. Case study: a smart-campus energy system

Imagine a campus with several buildings observed daily for two years.

For each building and day we record:

- temperature and humidity;
- occupancy;
- renewable-energy share;
- electricity use;
- carbon emissions;
- number of operational high-load incidents.

The data-generating mechanism intentionally contains four structures that motivate DSS5102 methods:

### A. Nonlinearity

Cooling and heating demand make the temperature effect approximately U-shaped:

$$
Energy \approx \beta_0 + \beta_1(T-24)^2 + \cdots
$$

### B. Grouped dependence

Buildings have different baselines. Measurements from the same building are therefore correlated.

### C. Temporal dependence

A campus-wide AR(1) latent load shock evolves as

$$
u_t = 0.72u_{t-1} + \epsilon_t.
$$

### D. Non-Gaussian outcomes

Peak-event indicators are binary and incident counts are counts. This motivates GLMs.

In [3]:
def simulate_smart_campus(cfg: Config) -> pd.DataFrame:
    rng = np.random.default_rng(cfg.seed)
    dates = pd.date_range("2024-01-01", periods=cfg.n_days, freq="D")
    buildings = [f"B{i+1}" for i in range(cfg.n_buildings)]

    day = np.arange(cfg.n_days)
    annual = np.sin(2 * np.pi * day / 365.25)
    weekly = np.sin(2 * np.pi * day / 7)

    temperature = 29.0 + 3.1 * annual + rng.normal(0, 1.1, cfg.n_days)
    humidity = 76.0 - 5.0 * annual + rng.normal(0, 3.0, cfg.n_days)
    renewable_share = np.clip(
        0.16 + 0.00018 * day + 0.035 * annual + rng.normal(0, 0.012, cfg.n_days),
        0.05,
        0.50,
    )

    # Shared latent temporal load shock: deliberately creates autocorrelation.
    latent = np.zeros(cfg.n_days)
    for t in range(1, cfg.n_days):
        latent[t] = 0.72 * latent[t-1] + rng.normal(0, 16)

    # Building-level heterogeneity: random intercept-like and sensitivity effects.
    b_intercept = dict(zip(buildings, rng.normal(0, 85, cfg.n_buildings)))
    b_temp_sens = dict(zip(buildings, rng.normal(0, 1.2, cfg.n_buildings)))
    b_occ_scale = dict(zip(buildings, rng.uniform(0.75, 1.25, cfg.n_buildings)))

    rows = []
    for t, date in enumerate(dates):
        is_weekend = int(date.dayofweek >= 5)
        base_occ = 820 * (0.42 if is_weekend else 1.0) * (1 + 0.05 * weekly[t])

        for b in buildings:
            occupancy = max(30, base_occ * b_occ_scale[b] + rng.normal(0, 55))
            temp_centered = temperature[t] - 24.0

            energy = (
                620
                + b_intercept[b]
                + 2.1 * occupancy
                + 8.8 * temp_centered**2
                + b_temp_sens[b] * temp_centered
                + 2.3 * humidity[t]
                - 390 * renewable_share[t]
                + latent[t]
                + rng.normal(0, 42)
            )
            energy = max(120, energy)

            # Grid carbon factor also evolves over time and has weekly variation.
            grid_factor = 0.39 - 0.00008 * t + 0.018 * np.cos(2 * np.pi * t / 7)
            emissions = energy * grid_factor * (1 - 0.55 * renewable_share[t]) + rng.normal(0, 18)
            emissions = max(10, emissions)

            # Count outcome for Poisson GLM demonstration.
            log_rate = -4.3 + 0.00062 * energy + 0.010 * max(humidity[t] - 75, 0)
            incident_count = rng.poisson(np.exp(np.clip(log_rate, -6, 1.7)))

            rows.append({
                "date": date,
                "building": b,
                "temperature_c": temperature[t],
                "humidity_pct": humidity[t],
                "renewable_share": renewable_share[t],
                "occupancy": occupancy,
                "energy_kwh": energy,
                "emissions_kg": emissions,
                "incident_count": incident_count,
                "is_weekend": is_weekend,
                "day_index": t,
            })

    df = pd.DataFrame(rows)
    # Binary response defined using an interpretable operational threshold.
    threshold = df["energy_kwh"].quantile(0.78)
    df["peak_event"] = (df["energy_kwh"] > threshold).astype(int)
    df["temp_sq"] = (df["temperature_c"] - 24.0) ** 2
    return df

panel = simulate_smart_campus(CFG)
panel.head()

,date,building,temperature_c,humidity_pct,renewable_share,occupancy,energy_kwh,emissions_kg,incident_count,is_weekend,day_index,peak_event,temp_sq
0,2024-01-01,B1,29.826516,70.86584,0.153175,976.888828,3220.729520,1218.322757,0,0,0,1,33.948284
1,2024-01-01,B2,29.826516,70.86584,0.153175,824.338194,2996.214065,1118.297120,0,0,0,1,33.948284
2,2024-01-01,B3,29.826516,70.86584,0.153175,688.439740,2540.824411,940.097762,0,0,0,0,33.948284
3,2024-01-01,B4,29.826516,70.86584,0.153175,731.905818,2401.087950,900.321930,0,0,0,0,33.948284
4,2024-01-01,B5,29.826516,70.86584,0.153175,793.884161,2661.997917,1015.197983,0,0,0,0,33.948284


In [4]:
print("Shape:", panel.shape)
print("Date range:", panel["date"].min().date(), "to", panel["date"].max().date())
print("Buildings:", panel["building"].nunique())
print("Peak-event rate:", round(panel["peak_event"].mean(), 3))
print("Mean incident count:", round(panel["incident_count"].mean(), 3))

display(panel.describe(include="number").T.round(3))

Shape: (5840, 13)
Date range: 2024-01-01 to 2025-12-30
Buildings: 8
Peak-event rate: 0.22
Mean incident count: 0.066


,count,mean,std,min,25%,50%,75%,max
temperature_c,5840.0,28.958,2.460,22.605,26.917,28.876,31.037,34.985
humidity_pct,5840.0,75.869,4.635,60.801,72.536,75.807,79.273,90.908
renewable_share,5840.0,0.226,0.038,0.146,0.194,0.227,0.258,0.304
occupancy,5840.0,682.052,253.540,105.627,410.357,738.958,888.782,1204.827
energy_kwh,5840.0,2427.536,600.459,907.824,1978.247,2515.663,2889.155,4265.228
emissions_kg,5840.0,768.656,202.807,238.055,618.762,780.942,913.854,1525.233
incident_count,5840.0,0.066,0.261,0.000,0.000,0.000,0.000,2.000
is_weekend,5840.0,0.285,0.451,0.000,0.000,0.000,1.000,1.000
day_index,5840.0,364.500,210.751,0.000,182.000,364.500,547.000,729.000
peak_event,5840.0,0.220,0.414,0.000,0.000,0.000,0.000,1.000


## 1.1 First visual diagnosis

Before fitting a model, we should ask what the data are trying to tell us.

The temperature-energy plot should reveal curvature. If we force a straight line through such a relationship, the model is misspecified even if the software returns coefficients without complaint.

In [35]:
sample = panel.sample(1800, random_state=CFG.seed).sort_values("temperature_c")

p = figure(
    width=900,
    height=380,
    title="Electricity demand versus temperature: visible nonlinearity",
    tools="pan,wheel_zoom,box_zoom,reset,save",
)
p.scatter(
    sample["temperature_c"],
    sample["energy_kwh"],
    size=5,
    alpha=0.22,
    legend_label="Building-days",
)
p.xaxis.axis_label = "Temperature (°C)"
p.yaxis.axis_label = "Energy (kWh/day)"
p.legend.location = "top_left"
show(p)

In [36]:
daily = (
    panel.groupby("date", as_index=False)
    .agg(
        energy_kwh=("energy_kwh", "sum"),
        emissions_kg=("emissions_kg", "sum"),
        temperature_c=("temperature_c", "mean"),
        humidity_pct=("humidity_pct", "mean"),
        renewable_share=("renewable_share", "mean"),
        occupancy=("occupancy", "sum"),
        incident_count=("incident_count", "sum"),
    )
)

show(make_time_plot(
    daily,
    "date",
    {"energy_kwh": "Campus energy"},
    "Campus electricity consumption over time",
    "Total kWh/day",
))

### What should you notice?

1. **Curvature** in energy versus temperature suggests that a simple linear term is inadequate.
2. **Repeated measurements within buildings** violate the idea that every row is an unrelated independent unit.
3. **Smooth temporal evolution** and repeated weekly/seasonal structure suggest autocorrelation.
4. Binary `peak_event` and count-valued `incident_count` require distributions other than a Gaussian response.

These observations determine the methods we need. Statistical modelling should be **problem-driven**, not algorithm-driven.

# 2. Baseline regression: establish a reference model

A baseline is essential. Sophisticated methods should earn their complexity by improving fit, prediction, assumptions or interpretation.

We begin with

$$
Energy_i = \beta_0 + \beta_1 Temperature_i + \beta_2 Humidity_i
+ \beta_3 Occupancy_i + \beta_4 RenewableShare_i + \epsilon_i.
$$

Because the true temperature relationship is nonlinear, we expect systematic residual structure.

In [37]:
ols = smf.ols(
    "energy_kwh ~ temperature_c + humidity_pct + occupancy + renewable_share",
    data=panel,
).fit()

print(ols.summary().tables[1])

                      coef    std err          t      P>|t|      [0.025      0.975]
-----------------------------------------------------------------------------------
Intercept       -1827.1144     63.945    -28.573      0.000   -1952.470   -1701.759
temperature_c      92.1792      0.984     93.722      0.000      90.251      94.107
humidity_pct        3.2755      0.524      6.255      0.000       2.249       4.302
occupancy           2.1426      0.007    309.686      0.000       2.129       2.156
renewable_share  -550.9196     48.153    -11.441      0.000    -645.317    -456.522


In [38]:
panel = panel.copy()
panel["ols_fitted"] = ols.fittedvalues
panel["ols_resid"] = ols.resid

res_sample = panel.sample(2200, random_state=CFG.seed)
p = figure(width=900, height=360, title="OLS residuals vs fitted values")
p.scatter(res_sample["ols_fitted"], res_sample["ols_resid"], size=5, alpha=0.25)
p.add_layout(Span(location=0, dimension="width", line_dash="dashed"))
p.xaxis.axis_label = "Fitted energy"
p.yaxis.axis_label = "Residual"
show(p)

If the residual cloud bends or fans out, that is not a cosmetic issue. It says:

> **The conditional mean or variance structure has not been captured adequately.**

A model can have a high $R^2$ and still be systematically wrong.

# 3. Generalized Linear Models (GLMs)

Ordinary regression assumes that a continuous response can be modelled with Gaussian errors. A GLM generalises this through three components:

1. a **random component**: the response distribution;
2. a **systematic component**: the linear predictor $\eta = X\beta$;
3. a **link function** $g$ connecting the mean to the linear predictor.

$$
g(E[Y\mid X]) = X\beta.
$$

We will study two canonical cases:

- logistic regression for a binary peak-load event;
- Poisson regression for incident counts.

## 3.1 Logistic GLM: probability of a peak-load event

Let

$$
Y_i = 1
$$

when a building-day crosses the operational peak-load threshold.

The logistic model is

$$
\log\frac{p_i}{1-p_i} = X_i\beta.
$$

Exponentiating a coefficient produces an **odds ratio**. If $e^{\beta_j}=1.20$, a one-unit increase in $X_j$ multiplies the odds by 1.20, holding other predictors fixed.

In [39]:
logit_glm = smf.glm(
    "peak_event ~ temperature_c + temp_sq + humidity_pct + occupancy + renewable_share + C(building)",
    data=panel,
    family=sm.families.Binomial(),
).fit()

coef = pd.DataFrame({
    "coefficient": logit_glm.params,
    "odds_ratio": np.exp(logit_glm.params),
    "p_value": logit_glm.pvalues,
}).round(4)

display(coef.loc[["temperature_c", "temp_sq", "humidity_pct", "occupancy", "renewable_share"]])

,coefficient,odds_ratio,p_value
temperature_c,-0.3197,0.7263,0.1837
temp_sq,0.4323,1.5408,0.0000
humidity_pct,0.0622,1.0642,0.0456
occupancy,0.0957,1.1005,0.0000
renewable_share,-20.0049,0.0000,0.0000


In [40]:
grid = pd.DataFrame({
    "temperature_c": np.linspace(panel.temperature_c.quantile(.01), panel.temperature_c.quantile(.99), 160),
})
grid["temp_sq"] = (grid["temperature_c"] - 24.0) ** 2
grid["humidity_pct"] = panel["humidity_pct"].median()
grid["occupancy"] = panel["occupancy"].median()
grid["renewable_share"] = panel["renewable_share"].median()
grid["building"] = "B1"
grid["peak_probability"] = logit_glm.predict(grid)

p = figure(width=900, height=350, title="Logistic GLM: predicted peak-event probability")
p.line(grid["temperature_c"], grid["peak_probability"], line_width=3)
p.xaxis.axis_label = "Temperature (°C)"
p.yaxis.axis_label = "P(peak event)"
show(p)

### Interpretation

The curve is not required to be linear in probability space. The linearity is on the **log-odds scale**.

Also notice why `temp_sq` matters. A positive quadratic effect allows extreme temperatures to increase peak-event probability even when the simple temperature coefficient alone would be misleading.

## 3.2 Poisson GLM: operational incident counts

For counts we use

$$
Y_i \sim Poisson(\lambda_i),
$$

with log link

$$
\log(\lambda_i) = X_i\beta.
$$

Exponentiating $\beta_j$ gives an **incidence-rate ratio**.

In [41]:
poisson_glm = smf.glm(
    "incident_count ~ energy_kwh + humidity_pct + renewable_share",
    data=panel,
    family=sm.families.Poisson(),
).fit()

pois_table = pd.DataFrame({
    "coefficient": poisson_glm.params,
    "incidence_rate_ratio": np.exp(poisson_glm.params),
    "p_value": poisson_glm.pvalues,
}).round(5)

display(pois_table)

pearson_dispersion = np.sum(poisson_glm.resid_pearson**2) / poisson_glm.df_resid
print("Pearson dispersion statistic:", round(float(pearson_dispersion), 3))

,coefficient,incidence_rate_ratio,p_value
Intercept,-3.55396,0.02861,0.00108
energy_kwh,0.00068,1.00068,0.00000
humidity_pct,-0.01167,0.98839,0.31667
renewable_share,-0.06586,0.93626,0.96230


Pearson dispersion statistic: 1.008


### Important diagnostic: overdispersion

For an ideal Poisson variable,

$$
Var(Y\mid X)=E[Y\mid X]=\lambda.
$$

A Pearson dispersion statistic much larger than 1 suggests **overdispersion**. In real applications, that may motivate a quasi-Poisson or Negative Binomial model.

This illustrates a DSS5102 habit: **fit → diagnose → revise**, rather than fit → report.

# 4. Bias–variance trade-off, shrinkage and cross-validation

Suppose we add polynomial features and building indicators. The feature space becomes richer, but richer models can become unstable.

Regularisation solves

### Ridge

$$
\hat\beta^{Ridge}
=
\arg\min_\beta
\left[
RSS + \lambda\sum_j\beta_j^2
\right].
$$

### Lasso

$$
\hat\beta^{Lasso}
=
\arg\min_\beta
\left[
RSS + \lambda\sum_j|\beta_j|
\right].
$$

As $\lambda$ grows:

- variance generally falls;
- bias generally rises;
- Ridge shrinks coefficients continuously;
- Lasso can set some coefficients exactly to zero.

The tuning parameter should be selected using **out-of-sample evidence**, not training error.

In [42]:
feature_cols = ["temperature_c", "humidity_pct", "occupancy", "renewable_share", "building"]
X = panel[feature_cols]
y = panel["energy_kwh"]

numeric = ["temperature_c", "humidity_pct", "occupancy", "renewable_share"]
categorical = ["building"]

preprocessor = ColumnTransformer([
    ("num", Pipeline([
        ("poly", PolynomialFeatures(degree=2, include_bias=False)),
        ("scale", StandardScaler()),
    ]), numeric),
    ("cat", OneHotEncoder(drop="first", handle_unknown="ignore"), categorical),
])

cv = KFold(n_splits=CFG.cv_splits, shuffle=True, random_state=CFG.seed)

def cv_rmse_for(model):
    pipe = Pipeline([("prep", preprocessor), ("model", model)])
    scores = cross_validate(
        pipe,
        X,
        y,
        cv=cv,
        scoring="neg_root_mean_squared_error",
        n_jobs=-1,
    )
    return -scores["test_score"].mean(), scores["test_score"].std()

alphas = np.logspace(-3, 3, 13)
rows = []
for a in alphas:
    r_mean, r_sd = cv_rmse_for(Ridge(alpha=a))
    l_mean, l_sd = cv_rmse_for(Lasso(alpha=a, max_iter=8000))
    rows.extend([
        {"model": "Ridge", "alpha": a, "cv_rmse": r_mean, "cv_sd": r_sd},
        {"model": "Lasso", "alpha": a, "cv_rmse": l_mean, "cv_sd": l_sd},
    ])

reg_path = pd.DataFrame(rows)
display(reg_path.sort_values("cv_rmse").head(8).round(3))

,model,alpha,cv_rmse,cv_sd
5,Lasso,0.010,46.879,0.297
3,Lasso,0.003,46.884,0.287
4,Ridge,0.010,46.886,0.312
2,Ridge,0.003,46.887,0.318
0,Ridge,0.001,46.887,0.320
1,Lasso,0.001,46.888,0.284
6,Ridge,0.032,46.890,0.294
7,Lasso,0.032,46.895,0.312


In [43]:
p = figure(
    width=900,
    height=370,
    x_axis_type="log",
    title="Cross-validated error versus regularisation strength",
)
for model_name in ["Ridge", "Lasso"]:
    d = reg_path[reg_path["model"] == model_name]
    p.line(d["alpha"], d["cv_rmse"], line_width=2.5, legend_label=model_name)
    p.scatter(d["alpha"], d["cv_rmse"], size=7, legend_label=model_name)
p.xaxis.axis_label = "alpha (log scale)"
p.yaxis.axis_label = "Mean CV RMSE"
p.legend.location = "top_left"
show(p)

### Why cross-validation matters

Training error usually decreases as flexibility increases. That does **not** prove generalisation improves.

$K$-fold CV approximates unseen-data performance:

$$
CV_K = \frac{1}{K}\sum_{k=1}^{K}L_k.
$$

The minimum of the CV curve estimates a useful complexity level. A very flat minimum also warns us not to over-interpret tiny differences between tuning parameters.

### Complexity

If a model fit costs approximately $T(n,p)$, $K$-fold cross-validation costs approximately

$$
\Theta(K\,T(n,p)).
$$

Hyperparameter search multiplies that cost by the number of candidates. This is why coarse-to-fine searches and parallel CV are valuable in production work.

# 5. Bootstrap: computational uncertainty quantification

Suppose we care about the association between renewable share and energy consumption.

Analytical standard errors rely on assumptions. The bootstrap instead approximates the estimator's sampling distribution by repeatedly resampling the observed data.

For bootstrap replicate $b$:

$$
D^{*(b)} \sim \text{sample with replacement from } D,
$$

and then

$$
\hat\beta^{*(b)} = T(D^{*(b)}).
$$

The empirical distribution of $\hat\beta^{*(b)}$ approximates estimator uncertainty.

In [44]:
boot_rng = np.random.default_rng(CFG.seed + 1)
boot_coefs = []
formula = "energy_kwh ~ temp_sq + humidity_pct + occupancy + renewable_share + C(building)"

for _ in range(CFG.bootstrap_reps):
    idx = boot_rng.integers(0, len(panel), len(panel))
    sample_b = panel.iloc[idx]
    fit_b = smf.ols(formula, data=sample_b).fit()
    boot_coefs.append(fit_b.params["renewable_share"])

boot_coefs = np.asarray(boot_coefs)
ci_low, ci_high = np.quantile(boot_coefs, [0.025, 0.975])

print("Bootstrap mean coefficient:", round(float(boot_coefs.mean()), 3))
print("Bootstrap standard error:", round(float(boot_coefs.std(ddof=1)), 3))
print("95% percentile interval:", (round(float(ci_low), 3), round(float(ci_high), 3)))

Bootstrap mean coefficient: -501.487
Bootstrap standard error: 15.628
95% percentile interval: (-529.995, -471.69)


In [45]:
hist, edges = np.histogram(boot_coefs, bins=30)
p = figure(width=900, height=350, title="Bootstrap distribution: renewable-share coefficient")
p.quad(top=hist, bottom=0, left=edges[:-1], right=edges[1:], alpha=0.55)
p.add_layout(Span(location=ci_low, dimension="height", line_dash="dashed", line_width=2))
p.add_layout(Span(location=ci_high, dimension="height", line_dash="dashed", line_width=2))
p.xaxis.axis_label = "Bootstrap coefficient"
p.yaxis.axis_label = "Frequency"
show(p)

### Caveat: dependence changes the bootstrap

The simple row bootstrap treats rows as exchangeable. Our data have repeated buildings and time dependence, so a real analysis should consider:

- cluster bootstrap by building;
- block bootstrap for time series;
- hierarchical/bootstrap schemes matching the sampling design.

This is a general principle:

> **A resampling scheme should respect the dependence structure of the data.**

# 6. Flexible nonlinear regression

The earlier scatterplot suggested that temperature does not act linearly.

A generic nonlinear regression writes

$$
Y = f(X) + \epsilon,
$$

where $f$ is estimated flexibly.

We compare:

1. linear regression;
2. polynomial regression;
3. spline regression;
4. LOWESS local regression;
5. a GAM-style additive smooth model.

## 6.1 Splines

A spline represents a nonlinear function using basis functions:

$$
f(x) = \beta_0 + \sum_{m=1}^{M}\beta_m B_m(x).
$$

The model remains linear in the coefficients even though it is nonlinear in $x$.

The key design question becomes **how flexible should the basis be?** Too few knots can underfit; too many can overfit unless regularised.

In [46]:
curve_df = panel.sample(3500, random_state=CFG.seed)[["temperature_c", "energy_kwh"]].sort_values("temperature_c")
X_temp = curve_df[["temperature_c"]]
y_temp = curve_df["energy_kwh"]

linear_temp = LinearRegression().fit(X_temp, y_temp)
poly_temp = Pipeline([
    ("poly", PolynomialFeatures(degree=2, include_bias=False)),
    ("lin", LinearRegression()),
]).fit(X_temp, y_temp)

spline_temp = Pipeline([
    ("spline", SplineTransformer(n_knots=8, degree=3, include_bias=False)),
    ("ridge", Ridge(alpha=1.0)),
]).fit(X_temp, y_temp)

grid_x = np.linspace(X_temp.temperature_c.min(), X_temp.temperature_c.max(), 220).reshape(-1, 1)

p = figure(width=900, height=390, title="Linear, polynomial and spline fits to the temperature effect")
p.scatter(curve_df["temperature_c"], curve_df["energy_kwh"], size=4, alpha=0.12, legend_label="Data")
p.line(grid_x[:, 0], linear_temp.predict(grid_x), line_width=2.5, legend_label="Linear")
p.line(grid_x[:, 0], poly_temp.predict(grid_x), line_width=2.5, legend_label="Quadratic")
p.line(grid_x[:, 0], spline_temp.predict(grid_x), line_width=3, legend_label="Cubic spline")
p.xaxis.axis_label = "Temperature (°C)"
p.yaxis.axis_label = "Energy (kWh/day)"
p.legend.location = "top_left"
p.legend.click_policy = "hide"
show(p)

### Interpretation

The linear model tries to summarise the entire relationship with one slope. The quadratic model can represent a U-shape, while a spline can adapt more locally.

More flexibility is not automatically better. Cross-validation should determine whether extra curvature generalises.

## 6.2 LOWESS: local regression

LOWESS estimates the curve around each target location using nearby observations with distance-dependent weights.

Conceptually, at $x_0$ it solves a local weighted regression:

$$
\hat f(x_0)
=
\arg\min_{\beta}
\sum_i w_i(x_0)
\left(y_i-\beta_0-\beta_1x_i\right)^2.
$$

The `frac` parameter is analogous to a bandwidth:

- smaller `frac` → more local/flexible;
- larger `frac` → smoother/more biased.

In [47]:
lowess_sample = panel.sample(2200, random_state=CFG.seed + 7).sort_values("temperature_c")
smoothed_18 = lowess(
    lowess_sample["energy_kwh"],
    lowess_sample["temperature_c"],
    frac=0.18,
    return_sorted=True,
)
smoothed_40 = lowess(
    lowess_sample["energy_kwh"],
    lowess_sample["temperature_c"],
    frac=0.40,
    return_sorted=True,
)

p = figure(width=900, height=380, title="LOWESS: effect of smoothing span")
p.scatter(lowess_sample["temperature_c"], lowess_sample["energy_kwh"], size=4, alpha=0.11, legend_label="Data")
p.line(smoothed_18[:, 0], smoothed_18[:, 1], line_width=3, legend_label="frac=0.18")
p.line(smoothed_40[:, 0], smoothed_40[:, 1], line_width=3, legend_label="frac=0.40")
p.xaxis.axis_label = "Temperature (°C)"
p.yaxis.axis_label = "Energy (kWh/day)"
p.legend.location = "top_left"
show(p)

## 6.3 GAM-style additive smooth modelling

A generalized additive model extends a linear predictor to smooth functions:

$$
g(E[Y]) = \beta_0 + f_1(X_1) + f_2(X_2) + \cdots + f_p(X_p).
$$

Here we construct a practical additive model using separate spline bases for temperature and humidity, with linear effects for occupancy and renewable share.

This retains an important advantage of GAMs:

> different predictors can have different nonlinear shapes while the overall model remains additive and interpretable.

In [48]:
gam_features = ["temperature_c", "humidity_pct", "occupancy", "renewable_share", "building"]
X_gam = panel[gam_features]
y_gam = panel["energy_kwh"]

additive_prep = ColumnTransformer([
    ("temp_spline", SplineTransformer(n_knots=8, degree=3, include_bias=False), ["temperature_c"]),
    ("hum_spline", SplineTransformer(n_knots=6, degree=3, include_bias=False), ["humidity_pct"]),
    ("linear", StandardScaler(), ["occupancy", "renewable_share"]),
    ("building", OneHotEncoder(drop="first", handle_unknown="ignore"), ["building"]),
])

gam_like = Pipeline([
    ("prep", additive_prep),
    ("model", Ridge(alpha=2.0)),
])

gam_like.fit(X_gam, y_gam)

cv_scores = cross_validate(
    gam_like,
    X_gam,
    y_gam,
    cv=cv,
    scoring="neg_root_mean_squared_error",
    n_jobs=-1,
)
print("GAM-style mean CV RMSE:", round(float(-cv_scores["test_score"].mean()), 3))

GAM-style mean CV RMSE: 47.713


In [49]:
temp_grid = pd.DataFrame({
    "temperature_c": np.linspace(panel.temperature_c.quantile(.01), panel.temperature_c.quantile(.99), 160),
    "humidity_pct": panel.humidity_pct.median(),
    "occupancy": panel.occupancy.median(),
    "renewable_share": panel.renewable_share.median(),
    "building": "B1",
})
temp_grid["prediction"] = gam_like.predict(temp_grid)

p = figure(width=900, height=350, title="GAM-style partial relationship: temperature")
p.line(temp_grid["temperature_c"], temp_grid["prediction"], line_width=3)
p.xaxis.axis_label = "Temperature (°C)"
p.yaxis.axis_label = "Predicted energy with other variables held typical"
show(p)

### What is a partial-effect plot doing?

We vary one predictor while holding the others at representative values. This is not a causal intervention. It is a way to inspect what the fitted model has learned conditionally.

The distinction is crucial:

$$
\text{association} \neq \text{causation}.
$$

# 7. Mixed-effects / multilevel modelling

Rows from the same building share unobserved properties: size, insulation, equipment efficiency, usage culture and so on.

A random-intercept mixed model writes

$$
Y_{ij}=X_{ij}\beta + u_j + \epsilon_{ij},
$$

where

$$
u_j \sim N(0,\tau^2),
\qquad
\epsilon_{ij}\sim N(0,\sigma^2).
$$

The fixed effects $\beta$ describe population-level relationships. The random effect $u_j$ represents building-specific deviation.

This produces **partial pooling**:

$$
\text{complete pooling}
\leftarrow
\text{partial pooling}
\rightarrow
\text{no pooling}.
$$

In [50]:
mixed_formula = "energy_kwh ~ temp_sq + humidity_pct + occupancy + renewable_share"

mixed = smf.mixedlm(
    mixed_formula,
    data=panel,
    groups=panel["building"],
).fit(reml=False, method="lbfgs")

print(mixed.summary())

              Mixed Linear Model Regression Results
Model:               MixedLM    Dependent Variable:    energy_kwh 
No. Observations:    5840       Method:                ML         
No. Groups:          8          Scale:                 2189.2060  
Min. group size:     730        Log-Likelihood:        -30778.5132
Max. group size:     730        Converged:             Yes        
Mean group size:     730.0                                        
------------------------------------------------------------------
                  Coef.   Std.Err.    z    P>|z|  [0.025   0.975] 
------------------------------------------------------------------
Intercept         671.696   42.239  15.902 0.000  588.908  754.483
temp_sq             9.013    0.032 284.947 0.000    8.951    9.075
humidity_pct        2.133    0.176  12.150 0.000    1.789    2.477
occupancy           2.098    0.003 803.123 0.000    2.093    2.104
renewable_share  -501.205   16.792 -29.847 0.000 -534.117 -468.293
Group Var 

In [51]:
random_intercepts = pd.DataFrame({
    "building": list(mixed.random_effects.keys()),
    "random_intercept": [float(np.asarray(v).ravel()[0]) for v in mixed.random_effects.values()],
}).sort_values("random_intercept")

display(random_intercepts.round(2))

p = figure(
    x_range=random_intercepts["building"].tolist(),
    width=900,
    height=350,
    title="Estimated building-specific random intercepts",
)
p.vbar(x=random_intercepts["building"], top=random_intercepts["random_intercept"], width=0.7)
p.add_layout(Span(location=0, dimension="width", line_dash="dashed"))
p.xaxis.axis_label = "Building"
p.yaxis.axis_label = "Deviation from population intercept"
show(p)

,building,random_intercept
3,B4,-155.16
4,B5,-79.31
7,B8,-73.10
6,B7,-54.78
2,B3,19.36
0,B1,36.96
5,B6,72.93
1,B2,233.10


In [52]:
var_group = float(np.asarray(mixed.cov_re).ravel()[0])
var_resid = float(mixed.scale)
icc = var_group / (var_group + var_resid)

print("Between-building variance:", round(var_group, 3))
print("Residual variance:", round(var_resid, 3))
print("Approximate ICC:", round(icc, 3))

Between-building variance: 12472.575
Residual variance: 2189.206
Approximate ICC: 0.851


## 7.1 Intraclass correlation (ICC)

A simple random-intercept ICC is

$$
ICC = \frac{\tau^2}{\tau^2+\sigma^2}.
$$

Interpretation:

> What fraction of residual variation is attributable to persistent differences between buildings?

A non-negligible ICC means the independence assumption of ordinary row-level regression is questionable.

### Advanced extension: random slopes

If buildings respond differently to temperature, use

$$
Y_{ij}=\beta_0+u_{0j}+(\beta_1+u_{1j})X_{ij}+\epsilon_{ij}.
$$

In `statsmodels`, that corresponds conceptually to `re_formula="~temperature_c"`. Random-slope models are richer but may require more groups and careful convergence diagnostics.

# 8. Transition from regression to time series

Regression asks how $Y$ changes with predictors. Time-series analysis adds another source of information:

> **the ordering of observations through time.**

We aggregate building-level data to daily campus totals.

The critical difference is that

$$
Y_t \not\perp Y_{t-1}.
$$

Yesterday's system state can help predict today's state.

In [53]:
ts = daily.set_index("date").asfreq("D")
train = ts.iloc[:-CFG.test_days].copy()
test = ts.iloc[-CFG.test_days:].copy()

print("Training observations:", len(train))
print("Test observations:", len(test))
print("Forecast horizon:", len(test), "days")

Training observations: 640
Test observations: 90
Forecast horizon: 90 days


## 8.1 Stationarity

Many classical time-series tools assume a stable stochastic process. Weak stationarity requires approximately constant mean and variance, with autocovariance depending on lag rather than absolute time.

The Augmented Dickey–Fuller test evaluates a null hypothesis of a unit root.

A small p-value is evidence **against** the unit-root null, but remember:

- stationarity is broader than one hypothesis test;
- seasonal patterns can still remain;
- visual diagnostics and subject-matter reasoning matter.

In [54]:
def adf_report(series, name):
    stat, pvalue, usedlag, nobs, crit, _ = adfuller(pd.Series(series).dropna(), autolag="AIC")
    return {
        "series": name,
        "ADF statistic": stat,
        "p_value": pvalue,
        "used_lag": usedlag,
        "n_obs": nobs,
        "5% critical": crit["5%"],
    }

adf_table = pd.DataFrame([
    adf_report(train["energy_kwh"], "energy level"),
    adf_report(train["energy_kwh"].diff(), "first difference"),
    adf_report(train["energy_kwh"].diff(7), "weekly difference"),
])
display(adf_table.round(4))

,series,ADF statistic,p_value,used_lag,n_obs,5% critical
0,energy level,-0.4191,0.9069,20,619,-2.8662
1,first difference,-8.1302,0.0000,20,618,-2.8662
2,weekly difference,-6.4917,0.0000,20,612,-2.8663


## 8.2 Autocorrelation and partial autocorrelation

The autocorrelation function is

$$
\rho_k=Corr(Y_t,Y_{t-k}).
$$

The PACF asks for the correlation at lag $k$ **after accounting for intermediate lags**.

Historically:

- ACF tailing off + PACF cutoff can suggest AR structure;
- PACF tailing off + ACF cutoff can suggest MA structure.

In modern practice, these are diagnostics and starting points rather than rigid identification rules.

In [55]:
show(gridplot([
    [acf_plot(train["energy_kwh"].diff().dropna(), 28, partial=False, title="ACF of first-differenced energy")],
    [acf_plot(train["energy_kwh"].diff().dropna(), 28, partial=True, title="PACF of first-differenced energy")],
]))

# 9. ARIMA modelling

An ARIMA$(p,d,q)$ model combines:

- $p$: autoregressive order;
- $d$: number of differences;
- $q$: moving-average order.

Using the backshift operator $B$,

$$
\phi(B)(1-B)^dY_t=\theta(B)\epsilon_t.
$$

A useful interpretation is:

$$
\boxed{\text{remove nonstationarity} + \text{model remaining serial dependence}.}
$$

We compare several candidate orders using AIC and out-of-sample forecast error.

In [56]:
arima_orders = [(1, 1, 1), (2, 1, 1), (1, 1, 2), (2, 1, 2), (3, 1, 1)]
arima_models = {}
arima_rows = []

for order in arima_orders:
    fit = ARIMA(train["energy_kwh"], order=order).fit()
    pred = fit.forecast(steps=len(test))
    name = f"ARIMA{order}"
    arima_models[name] = fit
    arima_rows.append({
        "model": name,
        "order": order,
        "AIC": fit.aic,
        "BIC": fit.bic,
        "test_RMSE": rmse(test["energy_kwh"], pred),
        "test_MAE": mean_absolute_error(test["energy_kwh"], pred),
    })

arima_compare = pd.DataFrame(arima_rows).sort_values("AIC").reset_index(drop=True)
display(arima_compare.round(2))

,model,order,AIC,BIC,test_RMSE,test_MAE
0,"ARIMA(2, 1, 2)","(2, 1, 2)",11984.58,12006.88,3846.16,3644.95
1,"ARIMA(3, 1, 1)","(3, 1, 1)",12165.00,12187.30,3879.27,3629.12
2,"ARIMA(2, 1, 1)","(2, 1, 1)",12223.23,12241.07,3868.85,3600.57
3,"ARIMA(1, 1, 2)","(1, 1, 2)",12232.40,12250.24,3844.10,3463.70
4,"ARIMA(1, 1, 1)","(1, 1, 1)",12356.52,12369.90,3842.86,3435.34


In [57]:
best_name = arima_compare.iloc[0]["model"]
best_arima = arima_models[best_name]
forecast_res = best_arima.get_forecast(steps=len(test))
forecast_mean = forecast_res.predicted_mean
forecast_ci = forecast_res.conf_int()

plot_df = pd.DataFrame({
    "date": test.index,
    "actual": test["energy_kwh"].values,
    "forecast": forecast_mean.values,
    "lower": forecast_ci.iloc[:, 0].values,
    "upper": forecast_ci.iloc[:, 1].values,
})

p = figure(x_axis_type="datetime", width=900, height=390, title=f"{best_name}: out-of-sample forecast")
p.line(train.index[-150:], train["energy_kwh"].iloc[-150:], line_width=2, legend_label="Train")
p.line(plot_df["date"], plot_df["actual"], line_width=2.5, legend_label="Actual test")
p.line(plot_df["date"], plot_df["forecast"], line_width=2.5, legend_label="Forecast")
source = ColumnDataSource(plot_df)
band = Band(base="date", lower="lower", upper="upper", source=source, level="underlay", fill_alpha=0.18)
p.add_layout(band)
p.xaxis.axis_label = "Date"
p.yaxis.axis_label = "Campus energy (kWh/day)"
p.legend.location = "top_left"
show(p)

## 9.1 Residual diagnostics

A successful time-series model should leave residuals resembling unpredictable noise.

We therefore inspect

$$
\hat\epsilon_t = Y_t - \hat Y_t.
$$

Strong remaining autocorrelation means the model left predictable temporal structure unused.

In [58]:
arima_resid = pd.Series(best_arima.resid).dropna()
show(acf_plot(arima_resid, nlags=28, title=f"Residual ACF — {best_name}"))

### AIC versus forecast error

AIC is an **in-sample likelihood-based information criterion**:

$$
AIC=-2\log L+2k.
$$

Test RMSE evaluates an actual forecasting task.

The model with minimum AIC need not have minimum test RMSE. This is not a contradiction: they answer related but different questions.

# 10. VAR: multivariate time-series dynamics

ARIMA models one principal series. A vector autoregression models several series jointly:

$$
\mathbf Y_t = \mathbf c + A_1\mathbf Y_{t-1}+\cdots+A_p\mathbf Y_{t-p}+\boldsymbol\epsilon_t.
$$

We use weekly log changes in energy and emissions:

$$
\Delta_7\log Y_t = \log Y_t - \log Y_{t-7}.
$$

This removes much of the scale and weekly-level structure while retaining dynamic co-movement.

> VAR coefficients encode predictive temporal association. They do **not**, by themselves, establish causality.

In [59]:
var_data = pd.DataFrame(index=train.index)
var_data["energy_wlogdiff"] = np.log(train["energy_kwh"]).diff(7)
var_data["emissions_wlogdiff"] = np.log(train["emissions_kg"]).diff(7)
var_data = var_data.dropna()

selector = VAR(var_data).select_order(maxlags=10)
print(selector.summary())

selected_lag = selector.aic
if selected_lag is None or int(selected_lag) < 1:
    selected_lag = 1
selected_lag = int(selected_lag)

var_fit = VAR(var_data).fit(selected_lag)
print("Selected VAR lag by AIC:", selected_lag)
print("VAR AIC:", round(float(var_fit.aic), 4))

 VAR Order Selection (* highlights the minimums)  
       AIC         BIC         FPE         HQIC   
--------------------------------------------------
0       -13.46      -13.44   1.432e-06      -13.45
1       -13.45      -13.40   1.448e-06      -13.43
2       -13.44      -13.37   1.459e-06      -13.41
3       -13.44      -13.34   1.450e-06      -13.40
4       -13.45      -13.32   1.441e-06      -13.40
5       -13.44      -13.29   1.452e-06      -13.38
6       -13.44      -13.25   1.458e-06      -13.37
7      -13.93*     -13.71*  8.961e-07*     -13.84*
8       -13.92      -13.68   9.033e-07      -13.82
9       -13.92      -13.65   9.002e-07      -13.82
10      -13.91      -13.61   9.079e-07      -13.80
--------------------------------------------------
Selected VAR lag by AIC: 7
VAR AIC: -13.9308


In [60]:
steps = 30
initial = var_data.values[-selected_lag:]
var_fc = var_fit.forecast(initial, steps=steps)
var_fc_df = pd.DataFrame(var_fc, columns=var_data.columns)
var_fc_df["date"] = pd.date_range(train.index[-1] + pd.Timedelta(days=1), periods=steps, freq="D")

p = figure(x_axis_type="datetime", width=900, height=350, title="VAR forecast: weekly log changes")
p.line(var_fc_df["date"], var_fc_df["energy_wlogdiff"], line_width=2.5, legend_label="Energy weekly log change")
p.line(var_fc_df["date"], var_fc_df["emissions_wlogdiff"], line_width=2.5, legend_label="Emissions weekly log change")
p.add_layout(Span(location=0, dimension="width", line_dash="dashed"))
p.yaxis.axis_label = "Forecast weekly log change"
p.legend.location = "top_left"
show(p)

### What VAR adds conceptually

In a univariate AR model, past energy predicts future energy.

In a two-variable VAR, future energy can depend on past energy **and** past emissions, while future emissions can depend on both histories.

That is useful for dynamically coupled systems such as:

- macroeconomic indicators;
- electricity load and prices;
- climate variables;
- pollution measures;
- demand across related regions.

# 11. State-space models and the Kalman-filter viewpoint

State-space models distinguish an unobserved system state from noisy observations.

### Transition equation

$$
\mathbf x_t = F\mathbf x_{t-1} + \mathbf w_t.
$$

### Observation equation

$$
Y_t = H\mathbf x_t + v_t.
$$

The state $\mathbf x_t$ might contain a latent level, trend or seasonal component.

The Kalman filter recursively performs two conceptual operations:

1. **predict** the next latent state;
2. **update** that prediction using the new observation.

We fit a structural model with a local linear trend and weekly seasonality.

In [61]:
ucm = UnobservedComponents(
    train["energy_kwh"],
    level="local linear trend",
    seasonal=7,
).fit(disp=False)

ucm_fc = ucm.get_forecast(steps=len(test))
ucm_mean = ucm_fc.predicted_mean
ucm_ci = ucm_fc.conf_int()

print("State-space AIC:", round(float(ucm.aic), 2))
print("State-space test RMSE:", round(rmse(test["energy_kwh"], ucm_mean), 2))

State-space AIC: 10742.93
State-space test RMSE: 1423.39


In [62]:
ucm_plot = pd.DataFrame({
    "date": test.index,
    "actual": test["energy_kwh"].values,
    "forecast": ucm_mean.values,
    "lower": ucm_ci.iloc[:, 0].values,
    "upper": ucm_ci.iloc[:, 1].values,
})

p = figure(x_axis_type="datetime", width=900, height=390, title="State-space forecast: local trend + weekly seasonal state")
p.line(train.index[-150:], train["energy_kwh"].iloc[-150:], line_width=2, legend_label="Train")
p.line(ucm_plot["date"], ucm_plot["actual"], line_width=2.5, legend_label="Actual test")
p.line(ucm_plot["date"], ucm_plot["forecast"], line_width=2.5, legend_label="State-space forecast")
source = ColumnDataSource(ucm_plot)
p.add_layout(Band(base="date", lower="lower", upper="upper", source=source, fill_alpha=0.18, level="underlay"))
p.legend.location = "top_left"
p.yaxis.axis_label = "Campus energy (kWh/day)"
show(p)

### ARIMA versus state-space thinking

ARIMA describes dependence largely through lagged observations and shocks.

State-space modelling asks a different conceptual question:

> Could the observations be noisy measurements of latent components that evolve over time?

Many familiar models can be written in state-space form. The framework becomes especially powerful with missing data, time-varying components and sequential updating.

# 12. Model selection and model averaging

Selecting one model ignores **model uncertainty**.

For candidate models $M_m$, Akaike weights are constructed from

$$
\Delta_m = AIC_m - \min_j AIC_j,
$$

$$
w_m = \frac{\exp(-\Delta_m/2)}{\sum_j\exp(-\Delta_j/2)}.
$$

A model-averaged forecast is

$$
\hat Y_t^{avg}=\sum_m w_m\hat Y_{t,m}.
$$

We combine the ARIMA candidates and the state-space model on the same forecast horizon.

In [63]:
forecast_bank = {}
selection_rows = []

for name, fit in arima_models.items():
    fc = np.asarray(fit.forecast(steps=len(test)), dtype=float)
    forecast_bank[name] = fc
    selection_rows.append({"model": name, "AIC": float(fit.aic)})

forecast_bank["StateSpace"] = np.asarray(ucm_mean, dtype=float)
selection_rows.append({"model": "StateSpace", "AIC": float(ucm.aic)})

selection = pd.DataFrame(selection_rows)
selection["delta_AIC"] = selection["AIC"] - selection["AIC"].min()
selection["akaike_weight"] = np.exp(-0.5 * selection["delta_AIC"])
selection["akaike_weight"] /= selection["akaike_weight"].sum()
selection = selection.sort_values("akaike_weight", ascending=False).reset_index(drop=True)

display(selection.round(4))

avg_fc = np.zeros(len(test))
for _, row in selection.iterrows():
    avg_fc += row["akaike_weight"] * forecast_bank[row["model"]]

benchmark_rows = []
for name, pred in forecast_bank.items():
    benchmark_rows.append(regression_metrics(test["energy_kwh"], pred, name))
benchmark_rows.append(regression_metrics(test["energy_kwh"], avg_fc, "AIC-weighted average"))
forecast_benchmark = pd.DataFrame(benchmark_rows).sort_values("RMSE").reset_index(drop=True)
display(forecast_benchmark.round(2))

,model,AIC,delta_AIC,akaike_weight
0,StateSpace,10742.9325,0.0000,1.0
1,"ARIMA(2, 1, 2)",11984.5829,1241.6504,0.0
2,"ARIMA(3, 1, 1)",12165.0003,1422.0678,0.0
3,"ARIMA(2, 1, 1)",12223.2262,1480.2937,0.0
4,"ARIMA(1, 1, 2)",12232.4050,1489.4724,0.0
5,"ARIMA(1, 1, 1)",12356.5230,1613.5905,0.0


,model,RMSE,MAE,R2,MAPE_%
0,AIC-weighted average,1423.39,1162.50,0.86,6.67
1,StateSpace,1423.39,1162.50,0.86,6.67
2,"ARIMA(1, 1, 1)",3842.86,3435.34,0.00,22.60
3,"ARIMA(1, 1, 2)",3844.10,3463.70,-0.00,22.67
4,"ARIMA(2, 1, 2)",3846.16,3644.95,-0.00,22.91
5,"ARIMA(2, 1, 1)",3868.85,3600.57,-0.01,23.04
6,"ARIMA(3, 1, 1)",3879.27,3629.12,-0.02,23.12


In [64]:
compare_plot = pd.DataFrame({
    "date": test.index,
    "actual": test["energy_kwh"].values,
    "model_average": avg_fc,
    "best_arima": np.asarray(forecast_mean, dtype=float),
    "state_space": np.asarray(ucm_mean, dtype=float),
})

p = figure(x_axis_type="datetime", width=900, height=390, title="Forecast comparison on held-out data")
p.line(compare_plot["date"], compare_plot["actual"], line_width=3, legend_label="Actual")
p.line(compare_plot["date"], compare_plot["best_arima"], line_width=2, legend_label=best_name)
p.line(compare_plot["date"], compare_plot["state_space"], line_width=2, legend_label="State-space")
p.line(compare_plot["date"], compare_plot["model_average"], line_width=2.5, line_dash="dashed", legend_label="AIC-weighted average")
p.legend.location = "top_left"
p.legend.click_policy = "hide"
p.yaxis.axis_label = "Campus energy (kWh/day)"
show(p)

### Important nuance

Model averaging does not guarantee lower RMSE on every test set. Its purpose is to acknowledge uncertainty about which model specification is best.

AIC weights are one principled likelihood-based approach. Alternatives include:

- cross-validation weights;
- stacking weights;
- Bayesian model averaging;
- forecast combinations based on historical performance.

# 13. A unified DSS5102 decision framework

The methods now fit into one coherent diagnostic tree.

| Data/problem structure | Statistical response |
|---|---|
| Continuous response, approximately linear | Linear regression |
| Binary response | Logistic GLM |
| Count response | Poisson / Negative Binomial GLM |
| Many/correlated predictors | Ridge / Lasso |
| Need tuning/model comparison | Cross-validation |
| Need computational uncertainty | Bootstrap |
| Nonlinear continuous relationship | Splines / local regression |
| Several nonlinear additive effects | GAM |
| Repeated observations within groups | Mixed-effects model |
| One temporally dependent series | ARIMA-family model |
| Several interacting time series | VAR |
| Latent evolving components | State-space model |
| Multiple plausible models | Model selection / averaging |

The deeper lesson is not memorising this table. It is learning to ask:

$$
\boxed{
\text{What stochastic structure does my data require?}
}
$$

# 14. Methodical workflow for a real project

A disciplined DSS5102-style workflow is:

### Step 1 — Define the estimand or forecast target

What exactly are you trying to explain, estimate or predict?

### Step 2 — Identify the response distribution

Continuous? Binary? Count? Proportion?

### Step 3 — Inspect functional form

Are effects plausibly linear? Do plots show curvature, thresholds or saturation?

### Step 4 — Identify dependence

Are observations grouped, repeated, spatially related or ordered through time?

### Step 5 — Build a simple baseline

Do not start with the most complex available model.

### Step 6 — Diagnose failure modes

Residual pattern? Overdispersion? Autocorrelation? Group-level heterogeneity?

### Step 7 — Increase complexity for a reason

Use splines because of nonlinearity; random effects because of clusters; ARIMA because of serial dependence.

### Step 8 — Validate out of sample

Use CV for approximately exchangeable observations and rolling/temporal validation for forecasts.

### Step 9 — Quantify uncertainty

Confidence intervals, bootstrap intervals, prediction intervals or forecast intervals.

### Step 10 — Communicate assumptions

Every model embeds assumptions. Good statistical work makes them visible.

# 15. Exercises for students

## Exercise 1 — GLM distribution choice

Suppose the response becomes the **fraction of a building's daily energy supplied by solar power**. Why would ordinary Poisson regression be inappropriate? What model family would you investigate?

## Exercise 2 — Overdispersion

Modify the simulation so `incident_count` has variance much larger than its mean. Refit the Poisson GLM and inspect the Pearson dispersion statistic. Then investigate a Negative Binomial GLM.

## Exercise 3 — Regularisation

Increase the polynomial degree from 2 to 4. Compare:

- unregularised linear regression;
- Ridge;
- Lasso.

Does training fit improve while CV performance deteriorates?

## Exercise 4 — Spline complexity

Evaluate `n_knots` in

$$
\{4,6,8,12,16\}.
$$

Use CV RMSE to determine whether extra flexibility is justified.

## Exercise 5 — Mixed effects

Fit separate OLS regressions for each building and compare their intercepts with the mixed-model random intercepts. Which estimates are more extreme? Explain **partial pooling**.

## Exercise 6 — Time-aware validation

Why is randomly shuffled $K$-fold CV inappropriate for a true forecasting problem? Replace it with expanding-window validation.

## Exercise 7 — ARIMA diagnostics

Fit an intentionally poor ARIMA$(0,1,0)$ model. Compare its residual ACF with the selected candidate.

## Exercise 8 — VAR transformations

Fit VAR directly on raw levels, then on weekly log differences. Compare diagnostics and explain why nonstationarity can create misleading relationships.

## Exercise 9 — State-space structure

Compare:

- local level;
- local linear trend;
- local linear trend + weekly seasonality.

Use both AIC and held-out RMSE.

## Exercise 10 — Model uncertainty

Construct weights from inverse validation RMSE instead of AIC. Compare the resulting ensemble forecast with the AIC-weighted average.

# 16. Solution sketches

### Exercise 1

A fraction is bounded between 0 and 1, whereas a Poisson response is a non-negative integer count. Depending on the data-generating process, investigate binomial modelling for successes/trials or beta-type modelling for continuous proportions.

### Exercise 2

When conditional variance greatly exceeds conditional mean, Poisson standard errors can be too optimistic. Negative Binomial models add an overdispersion parameter.

### Exercise 3

A degree-4 polynomial enlarges the feature space. Unregularised estimates can have higher variance. Ridge should stabilise coefficients; Lasso may also remove weak terms.

### Exercise 4

More knots reduce approximation bias but increase flexibility. Select based on validation performance, not visual smoothness alone.

### Exercise 5

Separate-building estimates use no pooling and can be extreme, especially with sparse groups. Mixed models shrink noisy group estimates toward the population mean.

### Exercise 6

Random CV can leak future information into training folds. Forecast evaluation should preserve chronological order.

### Exercise 7

If the residual ACF retains significant structure, the fitted process did not absorb all predictable serial dependence.

### Exercise 8

Persistent trending levels can make unrelated series appear strongly related. Differencing/log-differencing can help restore stationarity before VAR fitting.

### Exercise 9

A richer latent-state model should only be preferred when likelihood/forecast evidence supports the added state structure.

### Exercise 10

Different weighting schemes answer different optimisation goals. AIC weights emphasise relative information loss; validation weights emphasise empirical predictive performance.

# 17. Extensions beyond the central DSS5102 toolkit

Once the material here is comfortable, natural next topics include:

- ARIMAX / dynamic regression with exogenous predictors;
- intervention analysis and structural breaks;
- seasonal ARIMA;
- hierarchical forecasting;
- cointegration and vector error-correction models;
- stochastic volatility and ARCH/GARCH;
- Bayesian hierarchical models;
- conformal prediction for time series;
- gradient-boosted trees with lag features;
- recurrent neural networks, temporal CNNs and Transformers;
- modern time-series foundation models.

These are best understood **after** mastering the statistical structures in this notebook: nonlinear mean functions, dependence, resampling, regularisation, latent states and principled forecast evaluation.

# 18. Final conceptual map

DSS5102 can be remembered through four questions.

## Question A — What distribution does the response have?

$$
\text{Gaussian / Bernoulli / Poisson / ...}
\Rightarrow \text{GLM family and link}
$$

## Question B — What shape does the conditional relationship have?

$$
\text{linear} \rightarrow \text{polynomial} \rightarrow \text{spline / local / additive smooth}
$$

## Question C — What dependence exists between observations?

$$
\text{independent}
\rightarrow
\text{grouped/hierarchical}
\rightarrow
\text{temporally dependent}
$$

## Question D — How uncertain is our chosen model?

$$
\text{cross-validation}
+
\text{bootstrap}
+
\text{information criteria}
+
\text{model averaging}.
$$

The central skill is therefore not remembering which Python class fits which model. It is being able to look at a real data-generating problem and reason:

$$
\boxed{
\text{structure} \rightarrow \text{assumptions} \rightarrow \text{model} \rightarrow
\text{diagnostics} \rightarrow \text{validation} \rightarrow \text{uncertainty}
}
$$

That is the modelling mindset this notebook is designed to build.